In [0]:
conn_string = dbutils.secrets.get("pkustra555-scope", "pkustra555-eventhub-cs")
dbutils.widgets.text("login", "pkustra555")
dbutils.widgets.text("catalog", "dbr_dev")

login = dbutils.widgets.get("login")
catalog = dbutils.widgets.get("catalog")
eh_name = "pkustra555_evh"

In [0]:
dbutils.widgets.text("namespace_name", "evhpl24databricks")
namespace_name = dbutils.widgets.get("namespace_name")
bootstrap_servers= f"{namespace_name}.servicebus.windows.net:9093"
sasl_config =("kafkashaded.org.apache.kafka.common.security.plain."
        "PlainLoginModule required "
        'username="$ConnectionString" ' 
        f'password="{conn_string}";')


In [0]:
kafka_options = {
  "kafka.bootstrap.servers": bootstrap_servers,
  "subscribe": eh_name,
  "kafka.security.protocol": "SASL_SSL",
  "kafka.sasl.mechanism": "PLAIN",
  "kafka.sasl.jaas.config": sasl_config,
  "startingOffsets": "latest"
}

df_raw_stream = spark.readStream.format("kafka").options(**kafka_options).load()

In [0]:
df_raw_stream.printSchema()

In [0]:
from pyspark.sql import functions as F 
df_decoded = (df_raw_stream.select(
    F.col("value").cast("string").alias("event_json"),
    F.col("topic"),
    F.col("partition"),
    F.col("offset"),
    F.col("timestamp").alias("eventhub_timestamp")
))

In [0]:
bronze_schema = f"{login}_bronze"
checkpoint_path = f"/Volumes/{catalog}/{bronze_schema}/wikipedia_streaming/checpoints"
target_table = f"{catalog}.{bronze_schema}.wikipedia_streaming_bronze"

In [0]:
from pyspark.sql.functions import current_timestamp
df_bronze = (df_decoded.withColumn("ingested_at", current_timestamp()))

In [0]:
(df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table))

In [0]:
display(spark.table(target_table))

In [0]:
before_count = spark.table(target_table).count()
print(f"Before: {before_count}")

In [0]:
(df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table))

In [0]:
after_count = spark.table(target_table).count()
print(f"After: {after_count}")

I could use UDFs transormations but it wouldn't be as efficient as using Spark's built-in functions. 
UDF (user defined function) it is a custom function written by the user. I did not use UDFs because all required transformations could be implemented using Spark's built-in functions which are optimized.
Example of UDF:

```python
def user_type(name):
    if "bot" in name.lower():
        return "BOT"
    return "HUMAN"

user_type_udf = udf(user_type)

df = df.withColumn("user_type", user_type_udf("user"))
```